In [157]:
import os
import pandas as pd
import numpy as np
import re
import statistics

In [158]:
def transformar_con_ancla(ruta_archivo):
    try:
        df_raw = pd.read_excel(ruta_archivo, sheet_name=0, header=None)
        df_raw = df_raw.iloc[:, :60] 

        # 🔥 BLINDAJE 1: Rescatar el valor de la Región (RG) ANTES de cortar el archivo.
        # Propagamos la primera columna hacia abajo para que el texto de la Región no se pierda.
        df_raw.iloc[:, 0] = df_raw.iloc[:, 0].replace(["", "None", "nan", "NaN", "NAN"], np.nan).ffill()

        # Buscar el ancla
        fila_ancla = -1
        for i in range(15): 
            fila_texto = df_raw.iloc[i].fillna("").astype(str).str.upper().tolist()
            if any("UNIDAD" in celda for celda in fila_texto) or any("RG" in celda for celda in fila_texto):
                fila_ancla = i
                break
                
        if fila_ancla == -1: return None

        fila_sup = df_raw.iloc[fila_ancla].fillna("").astype(str).tolist()
        fila_med = df_raw.iloc[fila_ancla + 1].fillna("").astype(str).tolist()
        fila_inf = df_raw.iloc[fila_ancla + 2].fillna("").astype(str).tolist()

        nuevos_nombres = []
        sup_actual, med_actual = "", ""
        for i in range(len(df_raw.columns)):
            val_sup = str(fila_sup[i]).replace("nan", "").strip().upper()
            val_med = str(fila_med[i]).replace("nan", "").strip().upper()
            val_inf = str(fila_inf[i]).replace("nan", "").strip().upper()
            
            if val_sup and "UNNAMED" not in val_sup: sup_actual = val_sup
            if val_med and "UNNAMED" not in val_med: med_actual = val_med
                
            clean_sup = sup_actual if len(sup_actual) < 80 else f"BLOQUE_{1 if i < 35 else 2}"
            clean_med = med_actual if len(med_actual) < 80 else f"SUB_{i}"
            clean_inf = val_inf if len(val_inf) < 80 else ""
            
            partes = [p for p in [clean_sup, clean_med, clean_inf] if p]
            nombre_final = re.sub(r'\s+', ' ', "_".join(partes)).strip("_")
            nuevos_nombres.append(nombre_final if nombre_final else f"COL_{i}")

        # 🔥 BLINDAJE 2: Fijar las primeras 3 columnas para que jamás se crucen o dupliquen
        nuevos_nombres[0] = "RG"
        nuevos_nombres[1] = "UNIDAD"
        nuevos_nombres[2] = "UBICACIÓN SALAS"

        nombres_unicos = []
        conteo = {}
        for nom in nuevos_nombres:
            if nom not in conteo:
                nombres_unicos.append(nom)
                conteo[nom] = 1
            else:
                nombres_unicos.append(f"{nom}_{conteo[nom]}")
                conteo[nom] += 1

        df_datos = df_raw.iloc[fila_ancla + 3:].copy()
        df_datos.columns = nombres_unicos

        # Rellenar las celdas combinadas de identificación (solo hacia abajo)
        cols_id = df_datos.columns[:5]
        df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN", "NAN"], np.nan).infer_objects(copy=False).ffill()

        df_datos = df_datos.dropna(how='all').reset_index(drop=True)
        df_datos['ARCHIVO_ORIGEN'] = os.path.basename(ruta_archivo)
        return df_datos

    except Exception as e:
        print(f"❌ Error en {os.path.basename(ruta_archivo)}: {e}")
        return None

In [159]:
#Función para reparar nombres repetidos en un DataFrame
def reparar_columnas_duplicadas(df):
    nombres_vistos = {}
    nuevos_nombres = []
    
    for col in df.columns:
        # Convertimos a string por seguridad
        nombre_str = str(col) 
        
        if nombre_str not in nombres_vistos:
            nombres_vistos[nombre_str] = 0
            nuevos_nombres.append(nombre_str)
        else:
            nombres_vistos[nombre_str] += 1
            # Le agregamos el sufijo numérico para hacerlo único
            nombre_unico = f"{nombre_str}_{nombres_vistos[nombre_str]}"
            nuevos_nombres.append(nombre_unico)
            
    df.columns = nuevos_nombres
    return df

In [160]:
# Definimos la ruta exacta sincronizada de OneDrive a ruta local
ruta_base = r'C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases'

#Manejo de pats y librerías para acceder a la carpeta compartida de OneDrive
try:
    os.chdir(ruta_base)
    print(f"Directorio actual: {os.getcwd()}")
    
    # Listado de archivos en la carpeta compartida
    archivos = os.listdir('.')
    print("\nArchivos disponibles en la carpeta compartida:")
    for archivo in archivos:
        print(f"- {archivo}")        
except FileNotFoundError:
    print("Error: No se encontró la ruta. Asegúrate de que OneDrive esté iniciado.")

Directorio actual: C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases

Archivos disponibles en la carpeta compartida:
- .849C9593-D756-4E56-8D6E-42412F2A707B
- SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx
- SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 18092024 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx
- SALA DE DETENIDOS DÍA 22042025 IT. ACOSTA.xlsx
- SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx
- SALA DE DETENIDOS DÍA 30062024 IT. TOBÓN.xlsx
- SALA DE DETENIDOS DÍA 30092024 SI. GARCIA.xlsx
- SALA DE DETENIDOS

In [161]:
# ==========================================
# BUCLE DE LECTURA DE ARCHIVOS
# ==========================================
ruta_base = r'C:\Users\inter\Universidad Externado de Colombia\José William Cifuentes Sánchez - bases'
archivos = [f for f in os.listdir(ruta_base) if f.endswith(('.xlsx', '.xls'))]

todos_los_datos = []
print("--- 🚀 Iniciando Lectura de Archivos ---")
for archivo in archivos:
    df_temp = transformar_con_ancla(os.path.join(ruta_base, archivo))
    if df_temp is not None:
        todos_los_datos.append(df_temp)

# ==========================================
# FASE 2: CONSOLIDACIÓN Y LIMPIEZA FINAL
# ==========================================
if todos_los_datos:
    print("\n--- 🧩 Homologando Vocabulario y Uniendo la Base ---")
    
    for df in todos_los_datos:
        nombres_traducidos = []
        for col in df.columns:
            nombre = str(col).replace("RETENIDOS", "DETENIDOS").replace("SINDICADOS", "IMPUTADOS")
            nombre = nombre.replace("UBICIACIÓN", "UBICACIÓN").replace("CONDICION", "CONDICIÓN")
            nombre = nombre.replace("RETENIDAS", "DETENIDAS")
            if "_CONDICIÓNES MÉDICAS" in nombre and "CANTIDAD DE" not in nombre:
                nombre = nombre.replace("_CONDICIÓNES MÉDICAS", "_CANTIDAD DE CONDICIÓNES MÉDICAS")
            nombres_traducidos.append(nombre)
            
        nombres_unicos_finales = []
        conteo_nombres = {}
        for nom in nombres_traducidos:
            if nom not in conteo_nombres:
                nombres_unicos_finales.append(nom)
                conteo_nombres[nom] = 1
            else:
                nombres_unicos_finales.append(f"{nom}_{conteo_nombres[nom]}")
                conteo_nombres[nom] += 1
        df.columns = nombres_unicos_finales

    df_final = pd.concat(todos_los_datos, axis=0, ignore_index=True)
    
    col_rg = df_final.columns[0]
    col_unidad = df_final.columns[1] 
    
    # Destruye las filas de basura provenientes de Excel
    df_final = df_final.dropna(subset=[col_unidad])
    
    condicion_unidad = ~df_final[col_unidad].astype(str).str.contains('SUBTOTAL', case=False, na=False)
    condicion_rg = ~df_final[col_rg].astype(str).str.contains('TOTAL GENERAL', case=False, na=False)
    df_final = df_final[condicion_unidad & condicion_rg].reset_index(drop=True)
    
    # Asegurar que todas las columnas de métricas queden en Cero absoluto (sin NaN ni decimales)
    columnas_texto = [col_rg, col_unidad, 'UBICACIÓN SALAS', 'ARCHIVO_ORIGEN']
    cols_numericas = [c for c in df_final.columns if c not in columnas_texto]
    
    for col in cols_numericas:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0).astype(int)
    
    print(f"✨ ¡ÉXITO ABSOLUTO! Dataset purgado de NaN, filas fantasma y cruces de columnas.")
    print(f"📊 Registros reales: {df_final.shape[0]} | Columnas: {df_final.shape[1]}")
    display(df_final.head(10))
else:
    print("❌ No se encontraron datos para consolidar.")

--- 🚀 Iniciando Lectura de Archivos ---


C:\Users\inter\AppData\Local\Temp\ipykernel_20536\2921149437.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN", "NAN"], np.nan).infer_objects(copy=False).ffill()
C:\Users\inter\AppData\Local\Temp\ipykernel_20536\2921149437.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_datos[cols_id] = df_datos[cols_id].replace(["", "None", "nan", "NaN", "NAN"], np.nan).infer_objects(copy=False).ffill()
C:\Users\inter\AppData\Local\Temp\ipykernel_20536\2921149437.p


--- 🧩 Homologando Vocabulario y Uniendo la Base ---
✨ ¡ÉXITO ABSOLUTO! Dataset purgado de NaN, filas fantasma y cruces de columnas.
📊 Registros reales: 19340 | Columnas: 61


,RG,UNIDAD,UBICACIÓN SALAS,TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI_RG,TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI_RG,DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES,DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS,...,"DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1",ARCHIVO_ORIGEN,DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL,"DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,RG,MEBOG,CHAPINERO,0,0,21,296,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
1,RG,MEBOG,SANTAFÉ,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
2,RG,MEBOG,SAN CRISTÓBAL,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
3,RG,MEBOG,USME,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
4,RG,MEBOG,TUNJUELITO,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
5,RG,MEBOG,BOSA,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
6,RG,MEBOG,BOSA,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
7,RG,MEBOG,KENNEDY,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
8,RG,MEBOG,FONTIBÓN,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0
9,RG,MEBOG,ENGATIVÁ,0,0,0,0,0,0,0,...,0,0,0,0,0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,0,0,0,0


In [162]:
# Variables para guardar los DataFrames que vamos a comparar
df_ayala = None
df_otro = None
nombre_ayala = ""
nombre_otro = ""

# 1. Buscar los archivos en tu lista procesada
for df in todos_los_datos:
    origen = str(df['ARCHIVO_ORIGEN'].iloc[0]).upper()
    
    if "AYALA" in origen and df_ayala is None:
        df_ayala = df
        nombre_ayala = origen
    elif "AYALA" not in origen and df_otro is None:
        df_otro = df
        nombre_otro = origen
        
    # Si ya encontramos uno de cada uno, detenemos la búsqueda
    if df_ayala is not None and df_otro is not None:
        break

# 2. Hacer la comparación si se encontraron ambos
if df_ayala is not None and df_otro is not None:
    print("--- 📊 COMPARACIÓN DE ESTRUCTURAS ---")
    print(f"🔹 Archivo Ayala: {nombre_ayala} -> {len(df_ayala.columns)} columnas")
    print(f"🔸 Archivo Otro:  {nombre_otro} -> {len(df_otro.columns)} columnas\n")
    
    # Convertimos los nombres de las columnas a 'conjuntos' (sets) para compararlos fácilmente
    cols_ayala = set(df_ayala.columns)
    cols_otro = set(df_otro.columns)
    
    # Lo que tiene el archivo normal pero le falta al de Ayala
    faltan_en_ayala = cols_otro - cols_ayala
    print(f"❌ Columnas que ESTÁN en la plantilla normal pero FALTAN en Ayala ({len(faltan_en_ayala)}):")
    for c in sorted(faltan_en_ayala):
        print(f"   - {c}")
        
    # Lo que tiene Ayala pero NO está en el archivo normal (nombres distintos o inventados)
    sobran_en_ayala = cols_ayala - cols_otro
    print(f"\n⚠️ Columnas que ESTÁN en Ayala pero NO en la plantilla normal ({len(sobran_en_ayala)}):")
    for c in sorted(sobran_en_ayala):
        print(f"   - {c}")
        
else:
    print("❌ No se encontraron suficientes archivos en 'todos_los_datos' para hacer la comparación.")

--- 📊 COMPARACIÓN DE ESTRUCTURAS ---
🔹 Archivo Ayala: SALA DE DETENIDOS DÍA 23012024 IT. AYALA.XLSX -> 57 columnas
🔸 Archivo Otro:  SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.XLSX -> 61 columnas

❌ Columnas que ESTÁN en la plantilla normal pero FALTAN en Ayala (4):
   - DETENIDOS EN INSTALACIONES "URI"_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)
   - DETENIDOS EN INSTALACIONES "URI"_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)
   - DETENIDOS EN INSTALACIONES "URI"_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL
   - DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL

⚠️ Columnas que ESTÁN en Ayala pero NO en la plantilla normal (0):


In [163]:
print("--- 🔍 INICIO DEL DIAGNÓSTICO DE COLUMNAS ---")

archivos_con_problemas = 0

# Iteramos sobre la lista de DataFrames que ya tienes cargada
for i, df in enumerate(todos_los_datos):
    
    # 1. Evaluamos si el DataFrame tiene columnas con el mismo nombre
    if not df.columns.is_unique:
        archivos_con_problemas += 1
        
        # Intentamos obtener el nombre del archivo si guardamos el metadato, sino usamos el índice
        nombre_archivo = df['ARCHIVO_ORIGEN'].iloc[0] if 'ARCHIVO_ORIGEN' in df.columns else archivos[i]
        
        print(f"\n❌ ¡ALERTA! El archivo '{nombre_archivo}' tiene columnas DUPLICADAS.")
        
        # 2. Extraemos y mostramos exactamente cuáles son los nombres repetidos
        columnas_repetidas = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"   -> Nombres repetidos: {columnas_repetidas}")
        
        # 3. Mostramos cuántas veces se repite cada una para entender la gravedad
        for col_rep in columnas_repetidas:
            cantidad = (df.columns == col_rep).sum()
            print(f"      La columna '{col_rep}' aparece {cantidad} veces.")

# 4. Conclusión del diagnóstico
if archivos_con_problemas == 0:
    print("\n✅ Ningún DataFrame individual tiene columnas duplicadas.")
    print("El problema debe estar en la diferencia de cantidad de columnas entre archivos.")
    
    # Si no hay duplicados, imprimimos la cantidad de columnas de cada uno para comparar
    print("\n--- Auditoría de dimensiones ---")
    for i, df in enumerate(todos_los_datos):
        nombre_archivo = df['ARCHIVO_ORIGEN'].iloc[0] if 'ARCHIVO_ORIGEN' in df.columns else archivos[i]
        print(f"Archivo {i+1}: {nombre_archivo} -> {len(df.columns)} columnas")
else:
    print(f"\n⚠️ RESUMEN: Se encontraron {archivos_con_problemas} archivos con nombres de columnas repetidos.")

--- 🔍 INICIO DEL DIAGNÓSTICO DE COLUMNAS ---

✅ Ningún DataFrame individual tiene columnas duplicadas.
El problema debe estar en la diferencia de cantidad de columnas entre archivos.

--- Auditoría de dimensiones ---
Archivo 1: SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx -> 57 columnas
Archivo 2: SALA DE DETENIDOS DÍA - 12022025 IT. AYALA.xlsx -> 57 columnas
Archivo 3: SALA DE DETENIDOS DÍA - 23022025 IT. AYALA.xlsx -> 57 columnas
Archivo 4: SALA DE DETENIDOS DÍA - 28022025 IT. AYALA.xlsx -> 57 columnas
Archivo 5: SALA DE DETENIDOS DÍA - 29092024 IT. AYALA.xlsx -> 57 columnas
Archivo 6: SALA DE DETENIDOS DÍA - 31082024 IT. AYALA.xlsx -> 57 columnas
Archivo 7: SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 8: SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 9: SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx -> 61 columnas
Archivo 10: SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx -> 61 columnas
Archivo 11: SALA DE DETENIDOS DÍA 18092024 IT. 

In [164]:
print("--- 🛠️ Reparando archivos afectados por columnas repetidas ---")

# 2. Aplicamos la reparación directamente a tu lista existente 'todos_los_datos'
for i in range(len(todos_los_datos)):
    todos_los_datos[i] = reparar_columnas_duplicadas(todos_los_datos[i])

print("✅ Todos los DataFrames tienen columnas únicas ahora.")

# 3. Concatenación Final
print("\n--- 🚀 Iniciando Concatenación de Big Data ---")
try:
    # ignore_index=True resetea los números de fila del 0 en adelante
    df_consolidado = pd.concat(todos_los_datos, axis=0, ignore_index=True)
    
    print(f"📊 Registros consolidados: {df_consolidado.shape[0]} filas.")
    print(f"📊 Total de columnas resultantes: {df_consolidado.shape[1]} columnas.")
    
    # Vista previa
    display(df_consolidado.head())
    
except Exception as e:
    print(f"❌ Error inesperado: {e}")

--- 🛠️ Reparando archivos afectados por columnas repetidas ---
✅ Todos los DataFrames tienen columnas únicas ahora.

--- 🚀 Iniciando Concatenación de Big Data ---
📊 Registros consolidados: 19598 filas.
📊 Total de columnas resultantes: 61 columnas.


,RG,UNIDAD,UBICACIÓN SALAS,TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI_RG,TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI_RG,DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES,DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS,...,"DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1",ARCHIVO_ORIGEN,DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL,"DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,RG,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
1,RG,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
2,RG,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
3,RG,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
4,RG,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN


In [165]:
# Unimos todos los DataFrames de la lista en uno solo
if todos_los_datos:
    
    df_final = pd.concat(todos_los_datos, ignore_index=True)
    print(f"✅ Dataset consolidado inicial con {df_final.shape[0]} filas.")
    
    # Creamos una condición para excluir lo que diga 'SUBTOTAL' en 'UNIDAD'
    condicion_unidad = ~df_final['UNIDAD'].str.contains('SUBTOTAL', case=False, na=False)
    
    # Creamos una condición para excluir lo que diga 'TOTAL GENERAL' en 'RG'
    condicion_rg = ~df_final['RG'].str.contains('TOTAL GENERAL', case=False, na=False)
    
    # 3. Aplicamos los filtros al DataFrame y reseteamos el índice
    df_final = df_final[condicion_unidad & condicion_rg].reset_index(drop=True)
    
    print(f"🧹 Dataset limpio (sin subtotales ni totales): {df_final.shape[0]} filas y {df_final.shape[1]} columnas.")
    
    # Vista previa para confirmar
    display(df_final.head())

else:
    print("No hay datos para consolidar.")

✅ Dataset consolidado inicial con 19598 filas.
🧹 Dataset limpio (sin subtotales ni totales): 19340 filas y 61 columnas.


,RG,UNIDAD,UBICACIÓN SALAS,TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI_RG,TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI_RG,DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES,DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,DETENIDOS EN INSTALACIONES POLICIALES_CONDICIÓN_N° CONDENADOS,...,"DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1",ARCHIVO_ORIGEN,DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL,"DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,RG,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
1,RG,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
2,RG,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
3,RG,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN
4,RG,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,NaN,NaN,NaN,NaN


In [166]:
#Mover 'ARCHIVO_ORIGEN' al principio de la tabla para fácil lectura
columnas = df_final.columns.tolist()

# Verificamos que la columna exista para evitar errores
if 'ARCHIVO_ORIGEN' in columnas:
    # Sacamos la columna de donde esté y la insertamos en la posición 0
    columnas.insert(0, columnas.pop(columnas.index('ARCHIVO_ORIGEN')))
    
    # Reasignamos el orden al DataFrame
    df_final = df_final[columnas]
    print("✅ Columna 'ARCHIVO_ORIGEN' movida a la primera posición.")
else:
    print("⚠️ No se encontró la columna 'ARCHIVO_ORIGEN'. Revisa la función de transformación.")

#¿Cuántos datos aportó cada archivo?
print("\n--- 📊 Contraste de datos por archivo original ---")
conteo_archivos = df_final['ARCHIVO_ORIGEN'].value_counts()
print(conteo_archivos)

# Vista previa para confirmar que ahora es la primera columna
display(df_final.head(10))

✅ Columna 'ARCHIVO_ORIGEN' movida a la primera posición.

--- 📊 Contraste de datos por archivo original ---
ARCHIVO_ORIGEN
SALA DE DETENIDOS DÍA 12032025 PT. DÁVILA.xlsx     1088
SALA DE DETENIDOS DÍA 31032025 PT. DÁVILA.xlsx     1088
SALA DE DETENIDOS DÍA 20042025 PT.DÁVILA.xlsx      1088
SALA DE DETENIDOS DÍA 22042025 IT. ACOSTA.xlsx     1088
SALA DE DETENIDOS DÍA 17032025 IT. ACOSTA.xlsx     1088
SALA DE DETENIDOS DÍA 02032025 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 31102024 SI. GARCIA.xlsx     1085
SALA DE DETENIDOS DÍA 30092024 SI. GARCIA.xlsx     1085
SALA DE DETENIDOS DÍA 31012025 IT. ACOSTA.xlsx     1085
SALA DE DETENIDOS DÍA 30112024 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 18092024 IT. ACOSTA.xlsx     1085
SALA DE DETENIDOS DÍA 10032025 PT. DÁVILA.xlsx     1085
SALA DE DETENIDOS DÍA 31072024 IT. ACOSTA.xlsx     1046
SALA DE DETENIDOS DÍA 30062024 IT. TOBÓN.xlsx      1046
SALA DE DETENIDOS DÍA 27022024 IT. TOBÓN.xlsx      1042
SALA DE DETENIDOS DÍA - 29092024 IT. 

,ARCHIVO_ORIGEN,RG,UNIDAD,UBICACIÓN SALAS,TOTAL DE PERSONAS DETENIDAS EN INSTALACIONES POLICIALES Y URI_RG,TOTAL DEL PERSONAL QUE CUSTODIA LOS DETENIDOS EN INSTALACIONES POLICIALES Y URI_RG,DETENIDOS EN INSTALACIONES POLICIALES_CANTIDAD DE SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_CAPACIDAD DE LAS SALAS DEPTO O METRO,DETENIDOS EN INSTALACIONES POLICIALES_IMPUTADOS CON MAS DE 12 MESES,DETENIDOS EN INSTALACIONES POLICIALES_TOTAL DE PERSONAS EN LAS SALAS,...,"DETENIDOS EN INSTALACIONES ""URI""_# DE EXTRANJEROS (VENEZOLANOS)","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° CONDENADOS_1","DETENIDOS EN INSTALACIONES ""URI""_CONDICIÓN_N° IMPUTADOS_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_M_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_F_1","DETENIDOS EN INSTALACIONES ""URI""_GENERO_LGBTI_1",DETENIDOS EN INSTALACIONES POLICIALES_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL,"DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES)","DETENIDOS EN INSTALACIONES ""URI""_PARTE PERSONAL QUE CUSTODIA LOS DETENIDOS_TOTAL","DETENIDOS EN INSTALACIONES ""URI""_CANTIDAD DE CONDICIÓNES MÉDICAS (ESPECIALES CAPTURADOS VENEZOLANOS)"
0,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,CHAPINERO,NaN,NaN,21,296,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,SANTAFÉ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,SAN CRISTÓBAL,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,USME,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,TUNJUELITO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,BOSA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,BOSA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,KENNEDY,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,FONTIBÓN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SALA DE DETENIDOS DÍA 23012024 IT. AYALA.xlsx,RG,MEBOG,ENGATIVÁ,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [167]:
#ruta_proyecto = r"C:\Users\inter\OneDrive - Universidad Externado de Colombia\Tercer Semestre\Big Data\Proyecto_Final"
#os.chdir(ruta_proyecto)
#print("Nuevo cwd:", os.getcwd())
#os.makedirs("Resultados", exist_ok=True)
#df_final.to_excel("Resultados/dataset_consolidado_final.xlsx", index=False)